# Neutron Detection and Moderation


### Introduction
* This 'Jupyter Notebook' is your manual, data analysis and log book all at the same time. It consists of individual cells of text or Python code and you can edit or add cells to your heart's content.
* When using the notebook, there are two modes: *command* and *edit* mode. In command mode, you can add, move, run or delete cells. In edit mode, you modify its content. Use the *Esc* key or click outside the cell to leave edit mode. With *shift* + *return* you can run it directly from edit mode.
* When you 'run' a cell it will either be formated or executed depending on whether it is a text cell or a code cell, respectivly. The output will be shown inside the notebook.
* It is best to run the notebook piece-by-piece as we are progressing through the lab. If something doesn't work as expected, try 'Kernel->Restart' to get a fresh working environment.
* Code cells support typical editing features of an IDE, such as tab-complete. You find more info in the 'Help' menu including all keyboard shortcuts!
* **Please note**: The code is not supposed to work out-of-the-box but needs some tweaks from your side to complete all the way through! So if you encounter an error, don't despair, try to fix it and ask for help should you get stuck!
* **Tip**: If you enable line numbers for code cells in the menu (View->Toggle Line Numbers) it will be easier to discuss changes with your colleagues!


### Importing python packages <a name="import"></a>

In [ ]:
#This code cell holds useful code needed for the analysis. Execute it like normal.
# Packages to help importing files 
import sys, os
sys.path.append('./lib')

# Package that supports working with large arrays
import numpy as np  

# Package for plotting 
import matplotlib   # choose a backend for web applications; remove for stand-alone applications:
matplotlib.use('Agg') # enable interactive notebook plots (alternative: use 'inline' instead of 'notebook'/'widget' for static images)
%matplotlib ipympl

# The following line is the ONLY one needed in stand-alone applications!
import matplotlib.pyplot as plt

# Function that fits a curve to data 
from scipy.optimize import curve_fit

# Custom pakages prepared for you to analyze experimental data from labs.
# The code is located in the 'lib' subfolder which we have to specify:
sys.path.append('./lib')
import MCA, fittingFunctions

#  Measurement 1 - Base measurement<a name="load"></a>

Here we read in the data for the detectors in their default position, with no cadmium or aluminium plates inserted. 

## 1.1 Read in base measurements

Read in experimental data:

In [ ]:
#Load your data files here:
data_detector1 = MCA.load_spectrum(" ") #-> TODO <- Insert the name of your data file detector1
data_detector2 = MCA.load_spectrum(" ") #-> TODO <- Insert the name of your data file detector2

In [ ]:
#Execute this cell to plot your data

#### NO NEED TO EDIT ####

plt.figure()
plt.step(data_detector1.bin_centers, data_detector1.counts, where='mid')

plt.title("Detector1") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')

plt.figure()
plt.step(data_detector2.bin_centers, data_detector2.counts, where='mid')

plt.title("Detector2") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')
plt.show()

#plt.savefig("test_spectrum.png") # This is how you save the figure. Change the extension for different file types such as pdf or png.

## 1.2 Energy calibration of the detectors:

Recall that the energy calibration is the linear dependence between the ADC channel number and energy peaks of a spectrum. It should be performed with a linear regression.

In [ ]:
### Insert CHANNELS for the Detector1 
detector1_peak =          ### ToDo - insert value of the peak
detector1_ch_edge1 =      ### ToDo - insert value of the first edge 
detector1_ch_edge2 =      ### ToDo - insert value of the second edge 

### Insert CHANNELS for the Detector2
detector2_peak =         ### ToDo - insert value of the peak
detector2_ch_edge1 =     ### ToDo - insert value of the first edge 
detector2_ch_edge2 =     ### ToDo - insert value of the second edge 


### NO NEED TO EDIT ###
channels_1 = np.array([detector1_peak, detector1_ch_edge1, detector1_ch_edge2]) 
channels_2 = np.array([detector2_peak, detector2_ch_edge1, detector2_ch_edge2])

channels_empty = np.row_stack(([channels_1, channels_2]))
channels_empty

In [ ]:
energy = np.array([ , , ]) ### # -> TODO <- Insert the energies of the peak (Q value of reaction n+3He) and two edges (kinetic energy of alpha particle from rection going to excited state of 7Li,kinetic energy of alpha particle from rection going to ground state of 7Li)

A linear function looks like:
$$\text{Energy} = k \cdot \text{Channel} + m$$
where;
- $k$ determines the slope of the curve,
- $m$ determined the intersection of the y-axis.

In [ ]:
#Execute to find calibration coefficients for the two detectors.

#### NO NEED TO EDIT ####

k_arr = []
m_arr = []

### 2. the function
def LineFunc(x, k, m):
    return k*x+m

for i, channels in enumerate(channels_empty):

    ### 3. our intial guess 
    guess = [2, # k
             1] # m

    ### 4. now we can perform the fit using curve_fit:
    estimates, covar_matrix = curve_fit(LineFunc,
                                    channels,
                                    energy,
                                    p0 = guess)

    k_arr.append(estimates[0]) # unpack our values 
    m_arr.append(estimates[1]) # unpack our values 

    print("Estimates for Detector{}: k = {:.4f} and  m = {:.4f}".format(i+1, k_arr[i], m_arr[i]))

    ### Plot your Gaussian fit
    plt.figure()
    plt.plot(channels, energy, linestyle="", marker="*", label='data points')
    plt.plot(channels, LineFunc(channels, k_arr[i], m_arr[i]), label='linear fit')
    plt.legend(loc='upper left')
    plt.title("Energy calibration Detector{}".format(i+1)) # set title of the plot
    plt.xlabel("Channels")     # set label for x-axis 
    plt.ylabel("Energy [keV]")       # set label for y-axis 
    plt.show()

In [ ]:
#Execute to calibrate from channels to Energy [keV]

#### NO NEED TO EDIT ####
data_detector1.calibrate( k_arr[0], m_arr[0]) ### calibrate Detector1
data_detector2.calibrate( k_arr[1], m_arr[1]) ### calibrate Detector2
#########################

In [ ]:
#Execute this cell to plot your calibrated spectra of Detector1 and Detector2

#### NO NEED TO EDIT ####

plt.figure()
plt.step(data_detector1.energy, data_detector1.counts, where='mid', label='Calibrated data')
plt.title("Calibrated spectrum of Detector1") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts")          # set label for y-axis 
plt.yscale('log')
plt.legend()                  # enables legend for all plots with a 'label'

plt.figure()
plt.step(data_detector2.energy, data_detector2.counts, where='mid', label='Calibrated data')
plt.title("Calibrated spectrum of Detector2") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts")          # set label for y-axis 
plt.yscale('log')
plt.legend()                  # enables legend for all plots with a 'label'


## 1.3 Discussion points 
Make notes about the answer!

* **Question**: Identify what each region corresponds to.

# Measurement 2 - After addition of the aluminum sheet<a name="load"></a>


## 2.1 Read in spectra

In [ ]:
#Load your data files here:
data_detector1_aluminum = MCA.load_spectrum(" ")  ### Detector1 measurement with aluminum foil 
data_detector2_aluminum = MCA.load_spectrum(" ")  ### Detector2 measurement with aluminum foil 

In [ ]:
#Execute this cell to plot your spectra of Detector1 and Detector2 after adding the aluminum sheet

#### NO NEED TO EDIT ####

plt.figure()
plt.step(data_detector1_aluminum.bin_centers, data_detector1_aluminum.counts, where='mid')
plt.title("Detector1 with aluminum") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')

plt.figure()
plt.step(data_detector2_aluminum.bin_centers, data_detector2_aluminum.counts, where='mid')
plt.title("Detector2 with aluminum") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')
plt.show()

#plt.savefig("test_spectrum.png") # This is how you save the figure. Change the extension for different file types such as pdf or png.

## 2.2 Energy calibration 

We can use the previous calibration coefficients:

In [ ]:
#Execute to calibrate from ADC channels to Energy [keV]

### NO NEED TO EDIT ###
data_detector1_aluminum.calibrate( k_arr[0], m_arr[0]) ### Detector1 calibration
data_detector2_aluminum.calibrate( k_arr[1], m_arr[1]) ### Detector2 calibration 

## 2.3 Detector 1 comparison with and without the foil 

### Time normalization 

Here we normalize the counts per unit time.

In [ ]:
data_detector1.duration

In [ ]:
data_detector1_aluminum.duration

In [ ]:
counts_detector1_norm = data_detector1.counts / data_detector1.duration
counts_detector1_al_norm = data_detector1_aluminum.counts / data_detector1_aluminum.duration

In [ ]:
#Execute this cell to plot Detector1 before and after adding the aluminum sheet 

#### NO NEED TO EDIT ####

plt.figure()
plt.step(data_detector1.energy,counts_detector1_norm, where='mid', label='Detector1')
plt.step(data_detector1_aluminum.energy,counts_detector1_al_norm, where='mid', label='Detector1 with aluminum sheet')
plt.title("Detector1 with aluminum real time") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

### Find the ratio of the the Detector1 before and after the aluminum sheet

In [ ]:
#Execute to find the ratio of statistics before and after adding the aluminum sheet channel by channel

### Optional, you can change the max energy for which you plot the ratio  
E_cut = 1500 

### NO NEED TO EDIT ###
N_bins = len(data_detector1.energy)
ratio_al_det1 = np.zeros(N_bins)

for i in range(N_bins): 
    if (counts_detector1_norm[i]!=0 and counts_detector1_al_norm[i]!=0 and data_detector1.energy[i] < E_cut):
        ratio_al_det1[i] = counts_detector1_al_norm[i]/counts_detector1_norm[i]


In [ ]:
#Execute this cell to plot Detector1 before and after adding aluminum sheet and their ratio

#### NO NEED TO EDIT ####

plt.figure()
plt.step(data_detector1.energy,counts_detector1_norm, where='mid', label='Detector1')
plt.step(data_detector1_aluminum.energy,counts_detector1_al_norm, where='mid', label='Detector1 with aluminum sheet')
plt.step(data_detector1_aluminum.energy,ratio_al_det1, where='mid', label=f'Ratio')

plt.title("Detector1 with aluminum normalized to real time") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized with time")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Choose energy region to count the average ratio value
E_min =     ### ToDo: choose a good starting energy from the plot above 
E_max =     ### ToDo: choose a good end point from the plot above 

### NO NEED TO EDIT ###
ratio_sum = 0 
ratio_N = 0

for i in range(N_bins): 
    if (data_detector1.energy[i]>= E_min and data_detector1.energy[i]<=E_max):
        ratio_sum += ratio_al_det1[i]
        ratio_N += 1

ratio_avg_al_1 = ratio_sum/ratio_N

print("Average ratio of two measurements with and without the Aluminum foil for the region [{} {}] keV is {:.3f}".format(E_min, E_max, ratio_avg_al_1))

## 2.4 Detector 2 comparison with and without the foil 

### Time normalization 

Here we normalize the counts per unit time.

In [ ]:
data_detector2.duration

In [ ]:
data_detector2_aluminum.duration

In [ ]:
counts_detector2_norm = data_detector2.counts / data_detector2.duration
counts_detector2_al_norm = data_detector2_aluminum.counts / data_detector2_aluminum.duration

In [ ]:
#Execute this cell to plot Detector2 before and after adding the aluminum sheet 

### NO NEED TO EDIT ### 
plt.figure()
plt.step(data_detector2.energy,counts_detector2_norm, where='mid', label='Detector2')
plt.step(data_detector2_aluminum.energy,counts_detector2_al_norm, where='mid', label='Detector2 with aluminum sheet')
plt.title("Detector2 with aluminum real time") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

### Find the ratio of the the Detector2 before and after the aluminum sheet

In [ ]:
### Optional, you can change the max energy for which you plot the ratio  
E_cut = 1500 

### NO NEED TO EDIT ###
N_bins = len(data_detector1.energy)
ratio_al_det2 = np.zeros(N_bins)
for i in range(N_bins): 
    if (counts_detector2_norm[i]!=0 and counts_detector2_al_norm[i]!=0 and data_detector2.energy[i] < E_cut):
        ratio_al_det2[i] = counts_detector2_al_norm[i]/counts_detector2_norm[i]



In [ ]:
#Execute this cell to plot Detector2 before and after adding aluminum sheet and their ratio

#### NO NEED TO EDIT ####

plt.figure()
plt.step(data_detector2.energy,counts_detector2_norm, where='mid', label='Detector2')
plt.step(data_detector2_aluminum.energy,counts_detector2_al_norm, where='mid', label='Detector2 with aluminum sheet')
plt.step(data_detector2_aluminum.energy,ratio_al_det2, where='mid', label=f'Ratio')

plt.title("Detector2 with aluminum normalized to real time") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized with time")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Choose energy region to count the average ratio value
E_min =     ### To Do : choose a good starting energy from the plot above 
E_max =     ### To Do : choose a good end point from the plot above 

### NO NEED TO EDIT ###
ratio_sum = 0 
ratio_N = 0

for i in range(N_bins): 
    if (data_detector2.energy[i]>= E_min and data_detector2.energy[i]<=E_max):
        ratio_sum += ratio_al_det2[i]
        ratio_N += 1

ratio_avg_al_2 = ratio_sum/ratio_N

print("Average ratio of two measurements with and without the aluminum foil for the region [{} {}] keV is {:.3f}".format(E_min, E_max, ratio_avg_al_2))

## 2.5 Discussion points 
Make notes about the answer!

* **Question 1**: What are the effects of adding Aluminum foil for Detector 1? 

* **Question 2**: What are the effects of adding Aluminum foil on the Detector 2? 

# Measurement 3 - After addition of the cadmium sheet<a name="load"></a>

## 3.1 Read in spectra

Now load the spectra taken with the detectors after adding the cadmium sheet

In [ ]:
#Load your data files here:
data_detector1_cadmium = MCA.load_spectrum(" ")  ## ToDo: Inset name of the file with Detector1 with cadmium sheet measurement
data_detector2_cadmium = MCA.load_spectrum(" ")  ## ToDo: Inset name of the file with Detector2 with cadmium sheet measurement

In [ ]:
#Execute this cell to plot Detector1 and Detector2 after adding the cadmium sheet 

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector1_cadmium.bin_centers,data_detector1_cadmium.counts, where='mid')

plt.title("Detector1 with Cadmium") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')

plt.figure()
plt.step(data_detector2_cadmium.bin_centers, data_detector2_cadmium.counts, where='mid')
plt.yscale('log')

plt.title("Detector2 with Cadmium") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 

plt.show()

#plt.savefig("test_spectrum.png") # This is how you save the figure. Change the extension for different file types such as pdf or png.

## 3.2 Energy calibration 

We can use the previous calibration coefficients:

In [ ]:
#Execute to calibrate from channels to energy [keV]

#### NO NEED TO EDIT ####

data_detector1_cadmium.calibrate(k_arr[0], m_arr[0]) ### Detector1 with cadmium
data_detector2_cadmium.calibrate(k_arr[1], m_arr[1]) ### Detector2 with cadmium

## 3.3 Detector 1 comparison with and without the cadmium sheet 

### Time normalization 

Here we normalize the counts per unit time.

In [ ]:
data_detector1.duration

In [ ]:
data_detector1_cadmium.duration

In [ ]:
counts_detector1_cd_norm = data_detector1_cadmium.counts / data_detector1_cadmium.duration

In [ ]:
#Execute this cell to plot Detector1 before and after adding the cadmium sheet 

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector1.energy,counts_detector1_norm, where='mid', label='Detector1')
plt.step(data_detector1_cadmium.energy,counts_detector1_cd_norm, where='mid', label='Detector1 with cadmium sheet')
plt.title("Detector1 with Cadmium") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

### Find the ratio of the the Detector1 before and after the cadmium sheet

In [ ]:
### Optional, you can change the max energy for which you plot the ratio  
E_cut = 1500 

### NO NEED TO EDIT ###
N_bins = len(data_detector1.energy)
ratio_cd_det1 = np.zeros(N_bins)

for i in range(N_bins): 
    if (data_detector1.counts[i]!=0 and data_detector1_cadmium.counts[i]!=0 and data_detector1.energy[i] < E_cut):
        ratio_cd_det1[i] = counts_detector1_cd_norm[i]/counts_detector1_norm[i]


In [ ]:
#Execute this cell to plot Detector1 before and after adding the cadmium sheet with their ratio

### NO NEED TO EDIT ###
plt.figure()
plt.step(data_detector1.energy,counts_detector1_norm, where='mid', label='Detector1')
plt.step(data_detector1_cadmium.energy,counts_detector1_cd_norm, where='mid', label='Detector1 with Cadmium sheet')
plt.step(data_detector1_cadmium.energy,ratio_cd_det1, where='mid', label=f'Ratio')

plt.title("Detector1 with cadmium") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Choose energy region to count the average ratio value
E_min =     ### To Do : choose a good starting energy from the plot above 
E_max =     ### To Do : choose a good end point from the plot above 


### NO NEED TO EDIT ###

ratio_sum = 0 
ratio_N = 0

for i in range(N_bins): 
    if (data_detector1.energy[i]>= E_min and data_detector1.energy[i]<=E_max):
        ratio_sum += ratio_cd_det1[i]
        ratio_N += 1

ratio_avg_cd_1 = ratio_sum/ratio_N

print("Average ratio of two measurements with and without the cadmium sheet for the region [{} {}] keV is {:.3f}".format(E_min, E_max, ratio_avg_cd_1))

## 3.4 Detector 2 comparison with and without cadmium sheet 

### Time normalization 

Here we normalize the counts per unit time.

In [ ]:
data_detector2.duration

In [ ]:
data_detector2_cadmium.duration

In [ ]:
counts_detector2_cd_norm = data_detector2_cadmium.counts / data_detector2_cadmium.duration

In [ ]:
#Execute this cell to plot Detector2 before and after adding the cadmium sheet 

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector2.energy,counts_detector2_norm, where='mid', label='Detector2')
plt.step(data_detector2_cadmium.energy,counts_detector2_cd_norm, where='mid', label='Detector2 with cadmium sheet')
plt.title("Detector2 with Cadmium real time") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized with time")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

### Find the ratio of the the Detector2 before and after the cadmium sheet

In [ ]:
### Optional, you can change the max energy for which you plot the ratio  
E_cut = 1500 

### NO NEED TO EDIT ###
N_bins = len(data_detector1.energy)
ratio_cd_det2 = np.zeros(N_bins)
for i in range(N_bins): 
    if (counts_detector2_norm[i]!=0 and counts_detector2_cd_norm[i]!=0 and data_detector2.energy[i] < E_cut):
        ratio_cd_det2[i] = counts_detector2_cd_norm[i]/counts_detector2_norm[i]



In [ ]:
#Execute this cell to plot Detector2 before and after adding the cadmium sheet with their ratio

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector2.energy,counts_detector2_norm, where='mid', label='Detector2')
plt.step(data_detector2_cadmium.energy,counts_detector2_cd_norm, where='mid', label='Detector2 with cadmium sheet')
plt.step(data_detector2_cadmium.energy,ratio_cd_det2, where='mid', label=f'Ratio')

plt.title("Detector2 with cadmium normalized to real time") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Choose energy region to count the average ratio value
E_min =     ### To Do : choose a good starting energy from the plot above 
E_max =     ### To Do : choose a good end point from the plot above 


### NO NEED TO EDIT ###

ratio_sum = 0 
ratio_N = 0

for i in range(N_bins): 
    if (data_detector2.energy[i]>= E_min and data_detector2.energy[i]<=E_max):
        ratio_sum += ratio_cd_det2[i]
        ratio_N += 1

ratio_avg_cd_2 = ratio_sum/ratio_N

print("Average ratio of two measurements with and without the cadmium sheet for the region [{} {}] keV is {:.3f}".format(E_min, E_max, ratio_avg_cd_2))

## 3.5 Discussion points 
Make notes about the answer!

* **Question 1**: What are the effects of adding the cadmium sheet on Detector 1?
* **Question 2**: What are the effects of adding the cadmium sheet on Detector 2? 

# Measurement 4 - Change of detector position<a name="load"></a>

## 4.1 Read in spectra

Now load the new measurement with Detector2 in Position2:

In [ ]:
data_detector2_position2 = MCA.load_spectrum(" ")  ## ToDo: insert your filename

In [ ]:
#Execute this cell to plot Detector2 in the second position

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector2_position2.bin_centers,data_detector2_position2.counts, where='mid')

plt.title("Detector2 in Position2") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')


#plt.savefig("test_spectrum.png") # This is how you save the figure. Change the extension for different file types such as pdf or png.

## 4.2 Energy calibration 

We can use the previous calibration coefficients:

In [ ]:
#Execute to calibrate the new measurement

### NO NEED TO EDIT ### 
data_detector2_position2.calibrate( k_arr[1], m_arr[1])

## 4.3 Detector 2 comparison in Position 1 and Position 2

### Time normalization 

Here we normalize the counts per unit time.

In [ ]:
data_detector2.duration

In [ ]:
data_detector2_position2.duration

In [ ]:
counts_detector2_pos2_norm = data_detector2_position2.counts / data_detector2_position2.duration

In [ ]:
#Execute this cell to plot Detector2 in the position1 vs position2

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector2.energy,counts_detector2_norm, where='mid', label='Detector2')
plt.step(data_detector2_position2.energy,counts_detector2_pos2_norm, where='mid', label='Detector2 in Position2')
plt.title("Detector2 in Position2") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

### Find the ratio of the Detector2 in Position 1 and Position 2

In [ ]:
### Optional, you can change the max energy for which you plot the ratio  
E_cut = 1500 

### NO NEED TO EDIT ###
N_bins = len(data_detector2.energy)
ratio_det2_pos = np.zeros(N_bins)

for i in range(N_bins): 
    if (counts_detector2_norm[i]!=0 and counts_detector2_pos2_norm[i]!=0 and data_detector2.energy[i] < E_cut):
        ratio_det2_pos[i] = counts_detector2_pos2_norm[i]/counts_detector2_norm[i]


In [ ]:
#Execute this cell to plot Detector2 in the position1 vs position2 with their ratio

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector2.energy,counts_detector2_norm, where='mid', label='Detector2')
plt.step(data_detector2_position2.energy,counts_detector2_pos2_norm, where='mid', label='Detector2 in Position2')
plt.step(data_detector2_position2.energy,ratio_det2_pos, where='mid', label=f'Ratio')

plt.title("Detector2 in Position2") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts normalized")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Choose energy region to count the average ratio value
E_min =     ### ToDo : choose a good starting energy from the plot above 
E_max =     ### ToDo : choose a good end point from the plot above 

### NO NEED TO EDIT ###

ratio_sum = 0 
ratio_N = 0

for i in range(N_bins): 
    if (data_detector2.energy[i]>= E_min and data_detector2.energy[i]<=E_max):
        ratio_sum += ratio_det2_pos[i]
        ratio_N += 1

ratio_avg_pos = ratio_sum/ratio_N

print("Average ratio of two measurements in Position1 and Position2 for the region [{} {}] keV is {:.3f}".format(E_min, E_max, ratio_avg_pos))

## 4.4 Discussion points 
Make notes about the answer!

* **Question**: What are the effects of adding distance? Does it agree with your expectation from the inverse square law? Why/why not?

# Measurement 5: BGO spectrum

## 5.1 Load spectra
Note that the calibration coefficients are already given to you 

In [ ]:
data_BGO = MCA.load_spectrum(" ") ### ToDo: Insert the name of BGO spectrum

In [ ]:
# Execute to plot uncalibrated BGO spectrum

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_BGO.bin_centers,data_BGO.counts, where='mid')

plt.title("Uncalibrated BGO gamma spectrum") # set title of the plot
plt.xlabel("Channels")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')

plt.show()

## 5.2 Calibrate the BGO spectrum using calibration coefficients

In [ ]:
data_BGO.calibrate( , ) ### ToDo: Insert calibration values from the .Spe file values under $ENER_FIT 

In [ ]:
# Execute to plot calibrated BGO spectrum

### NO NEED TO EDIT ### 
plt.figure()
plt.step(data_BGO.energy,data_BGO.counts, where='mid')

plt.title("Calibrated BGO gamma spectrum") # set title of the plot
plt.xlabel("Energy, keV")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')

plt.show()
#plt.savefig("BGO_calibrated.png") # This is how you save the figure. Change the extension for different file types such as pdf or png.

## 5.3 Discussion points 
Discuss in a group and answer the following questions:

* **Question**: Where are gamma rays emitted from? 
* **Question**: What is the energy of the "bump" (around 2.2 MeV)? What are the smaller peaks to the left of it?

# Measurement 6: Li branching ratio calculation

**Note: This measurement is mandatory for FYSC22 only.**

## 6.1 Sum spectra

For this task we want to sum statistics for all measurements that we did with Detector1: base measurement + measurement with aluminum sheet + measurement with cadmium sheet.

In [ ]:
# Execute this cell to sum statistics of all measurements with the Detector1

### NO NEED TO EDIT ###
### Sum statistics of all measurements 
counts_sum = (
    data_detector1.counts
    + data_detector1_aluminum.counts
    + data_detector1_cadmium.counts
)

In [ ]:
# Execute this cell plot the base measurement vs all three measurements together with Detector1

### NO NEED TO EDIT ### 

plt.figure()
plt.step(data_detector1.energy,data_detector1.counts, where='mid', label='Detector1')
plt.step(data_detector1.energy,counts_sum, where='mid', label='Detector1 summed')

plt.title("Detector1 summed statistics") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
#plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

## 6.2 Compute the branching ratio numerically

In [ ]:
# Choose energy region to count the average ratio value
### Select good energy regio from the histogram 
E_min =    ### ToDo : choose a good starting energy from the plot above 
E_max =    ### ToDo : choose a good end point from the plot above 

E_replace_total =    ### Select the max E for which counts need to be "replaced" for the total spectrum
A_replace_total =    ### Select counts value value with which to replace 

E_replace_ground =  ### Select the max E for which counts need to be "replaced" for the ground state spectrum
A_replace_ground =  ### Select counts value value with which to replace 

### NO NEED TO EDIT ### 

N_bins = len(data_detector1.energy)
counts_total = np.zeros(N_bins)
counts_ground = np.zeros(N_bins)

counts_total_sum = 0 
counts_ground_sum = 0

for i in range(N_bins): 
    if (data_detector1.energy[i]>= E_min and data_detector1.energy[i]<=E_max):
        ### total 
        if (data_detector1.energy[i] < E_replace_total):
            counts_total[i] = A_replace_total
        else: 
            counts_total[i] = data_detector1.counts[i]
        ### ground state 
        if (data_detector1.energy[i] < E_replace_ground):
            counts_ground[i] = A_replace_ground
        else: 
            counts_ground[i] = data_detector1.counts[i]
        counts_total_sum += counts_total[i]
        counts_ground_sum += counts_ground[i]

In [ ]:
# Execute to plot population of the excited and ground state 

### NO NEED TO EDIT ### 
plt.figure()
plt.step(data_detector1.energy,data_detector1.counts, where='mid', label='Detector1 counts summed')
plt.step(data_detector1.energy,counts_total, where='mid', label='Population of excited state', color="orange")
plt.step(data_detector1.energy,counts_ground, where='mid', label='Population of ground state', color="green")

plt.fill_between(data_detector1.energy,counts_total, step="mid", alpha=0.4, color="orange")
plt.fill_between(data_detector1.energy,counts_ground, step="mid", alpha=0.4, color="green")


plt.title("Detector1") # set title of the plot
plt.xlabel("Reaction product energy, keV")     # set label for x-axis 
plt.ylabel("Counts")       # set label for y-axis 
plt.yscale('log')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Execute to find the ratio

### NO NEED TO EDIT ### 

print("Sum of the counts total {}".format(counts_total_sum))
print("Sum of the counts ground state {}".format(counts_ground_sum))

ratio_ground_total = counts_ground_sum / counts_total_sum

print("The ratio {:.2f}".format(ratio_ground_total))